In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("data_processed_credit_data_features_v2.csv")


In [42]:
y = df["default_flag"]


In [43]:
feature_cols = [
    "debt_to_income",
    "utilization_stress",
    "exposure_ratio",
    "avg_payment_delay",
    "max_payment_delay",
    "recent_delinquency_flag",
    "payment_to_bill_ratio",
    "missed_payment_count",
    "employment_stability",
    "has_past_default",
    "stress_interaction",
    "annual_income_missing_flag",
    "credit_utilization_missing_flag"
]

X = df[feature_cols]


In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)


In [45]:
X.isna().sum().sort_values(ascending=False)


payment_to_bill_ratio              263
debt_to_income                       0
utilization_stress                   0
exposure_ratio                       0
avg_payment_delay                    0
max_payment_delay                    0
recent_delinquency_flag              0
missed_payment_count                 0
employment_stability                 0
has_past_default                     0
stress_interaction                   0
annual_income_missing_flag           0
credit_utilization_missing_flag      0
dtype: int64

In [46]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)


In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)


In [48]:
model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="lbfgs"
)

model.fit(X_train_scaled, y_train)


LogisticRegression(class_weight='balanced', max_iter=1000)

In [49]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]


In [50]:
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc


0.9569856666666666

In [51]:
confusion_matrix(y_test, y_pred)


array([[19499,  2501],
       [  259,  2741]], dtype=int64)

In [52]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.99      0.89      0.93     22000
           1       0.52      0.91      0.67      3000

    accuracy                           0.89     25000
   macro avg       0.75      0.90      0.80     25000
weighted avg       0.93      0.89      0.90     25000



In [53]:
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

coef_df


,feature,coefficient
3,avg_payment_delay,3.914228
9,has_past_default,1.380256
1,utilization_stress,1.097734
0,debt_to_income,0.442566
2,exposure_ratio,0.194436
10,stress_interaction,0.175055
6,payment_to_bill_ratio,0.015009
8,employment_stability,-0.011131
7,missed_payment_count,-0.045682
5,recent_delinquency_flag,-0.082731


In [54]:
df1 = pd.read_csv("data_processed_credit_data_features_v2.csv")

In [55]:
# Assume y_prob is from model.predict_proba(X_test_scaled)[:,1]
# and X_test has index from original df

df1["pd_score"] = np.nan
df1.loc[X_test.index, "pd_score"] = y_prob

# Fill remaining PDs (Power BI requires no blanks)
df1["pd_score"] = df1["pd_score"].fillna(df1["pd_score"].median())


In [56]:
def risk_band(pd):
    if pd >= 0.60:
        return "High Risk"
    elif pd >= 0.35:
        return "Medium Risk"
    else:
        return "Low Risk"

df1["risk_band"] = df1["pd_score"].apply(risk_band)


In [58]:
fact_cols = [
    "customer_id",
    "default_flag",
    "pd_score",
    "risk_band",           # ✅ added
    "loan_amount",
    "credit_limit",
    "debt_to_income",
    "exposure_ratio",
    "utilization_stress",
    "avg_payment_delay",
    "max_payment_delay",
    "payment_to_bill_ratio",
    "missed_payment_count"
]

Fact_Credit_Risk = df1[fact_cols]
Fact_Credit_Risk.to_csv("Fact_Credit_Risk.csv", index=False)


In [59]:
dim_customer_cols = [
    "customer_id",
    "age",
    "gender",
    "education_level",
    "marital_status",
    "employment_years",
    "employment_stability"
]

Dim_Customer = df1[dim_customer_cols]
Dim_Customer.to_csv("Dim_Customer.csv", index=False)


In [60]:
dim_fin_cols = [
    "customer_id",
    "annual_income",
    "monthly_income",
    "credit_utilization",
    "credit_limit"
]

Dim_Financial_Profile = df1[dim_fin_cols]
Dim_Financial_Profile.to_csv("Dim_Financial_Profile.csv", index=False)


In [61]:
def risk_band(pd):
    if pd >= 0.60:
        return "High Risk"
    elif pd >= 0.35:
        return "Medium Risk"
    else:
        return "Low Risk"

Dim_Risk_Band = df1[["pd_score"]].copy()
Dim_Risk_Band["risk_band"] = Dim_Risk_Band["pd_score"].apply(risk_band)
Dim_Risk_Band = Dim_Risk_Band[["risk_band"]].drop_duplicates().reset_index(drop=True)
Dim_Risk_Band["risk_band_id"] = Dim_Risk_Band.index + 1

Dim_Risk_Band.to_csv("Dim_Risk_Band.csv", index=False)
